In [2]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv026931/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [3]:
from pyspark.sql.functions import *

In [4]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header","true") \
.option("inferschema","true") \
.load("/user/itv026931/lendingclubproject/raw/loans_defaulters_csv")

In [37]:
loans_def_raw_df

member_id,delinq_2yrs,delinq_amnt,pub_rec,pub_rec_bankruptcies,inq_last_6mths,total_rec_late_fee,mths_since_last_delinq,mths_since_last_record
3a366efe16c243e91...,0.0,0.0,1.0,1.0,0.0,0.0,null,83.0
9f3b33873233b9a42...,1.0,0.0,0.0,0.0,0.0,0.0,12.0,null
81d1bd5c408965373...,2.0,0.0,0.0,0.0,1.0,0.0,20.0,null
3ece6a3c9c20d41b9...,0.0,0.0,0.0,0.0,0.0,0.0,35.0,null
51686a0cbb9cfc4a4...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
34ee3c8e3bd6e25a4...,0.0,0.0,1.0,1.0,0.0,0.0,null,71.0
75ca7ec64ad50dc30...,0.0,0.0,0.0,0.0,0.0,0.0,74.0,null
c853bf125194e4998...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
fc5399a74637cfeb9...,0.0,0.0,1.0,1.0,0.0,0.0,41.0,104.0
0982b107c47ae1c72...,0.0,0.0,0.0,0.0,0.0,0.0,33.0,null


In [38]:
loans_def_raw_df.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- delinq_2yrs: string (nullable = true)
 |-- delinq_amnt: double (nullable = true)
 |-- pub_rec: string (nullable = true)
 |-- pub_rec_bankruptcies: double (nullable = true)
 |-- inq_last_6mths: string (nullable = true)
 |-- total_rec_late_fee: string (nullable = true)
 |-- mths_since_last_delinq: string (nullable = true)
 |-- mths_since_last_record: string (nullable = true)



In [40]:
loans_def_raw_df.createOrReplaceTempView("loan_defaulters")

Since the values of "deling_2yrs" is not in expected format, changing it's datatype to float and later to int

In [63]:
loans_def_raw_df.select("delinq_2yrs").distinct()

delinq_2yrs
1.0
271 monthly payme...
I bike to work on...
183xx
VISA and AMEX cre...
etc. and I feel t...
AZ
017xx
923xx
446xx


In [41]:
loans_def_raw_df.select(countDistinct("delinq_2yrs").alias("distinct_count"))

distinct_count
249


In [42]:
loans_def_raw_df.groupBy("delinq_2yrs").agg(count("*").alias("total")).orderBy(desc("total")).show(40)

+------------------+-------+
|       delinq_2yrs|  total|
+------------------+-------+
|               0.0|1838878|
|               1.0| 281335|
|               2.0|  81285|
|               3.0|  29539|
|               4.0|  13179|
|               5.0|   6599|
|               6.0|   3717|
|               7.0|   2062|
|               8.0|   1223|
|               9.0|    818|
|              10.0|    556|
|              11.0|    363|
|              12.0|    264|
|              13.0|    165|
|              14.0|    120|
|              15.0|     87|
|              null|     63|
|              16.0|     55|
|              17.0|     30|
|              18.0|     30|
|              19.0|     23|
|              20.0|     17|
|              21.0|     12|
|                CA|      8|
|                TX|      6|
|debt_consolidation|      5|
|              22.0|      5|
|    small_business|      5|
|                IL|      5|
|                FL|      4|
|              24.0|      4|
|             

In [43]:
spark.sql("select delinq_2yrs, count(*) as total_count from loan_defaulters group by delinq_2yrs order by total_count desc")

delinq_2yrs,total_count
0.0,1838878
1.0,281335
2.0,81285
3.0,29539
4.0,13179
5.0,6599
6.0,3717
7.0,2062
8.0,1223
9.0,818


In [5]:
loans_def_raw_schema = """member_id string, delinq_2yrs float, delinq_amnt float, pub_rec float, pub_rec_bankruptcies float,
inq_last_6mths float, total_rec_late_fee float, mths_since_last_delinq float, mths_since_last_record float"""

In [6]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header","true") \
.schema(loans_def_raw_schema) \
.load("/user/itv026931/lendingclubproject/raw/loans_defaulters_csv")

In [8]:
loans_def_raw_df.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- delinq_2yrs: float (nullable = true)
 |-- delinq_amnt: float (nullable = true)
 |-- pub_rec: float (nullable = true)
 |-- pub_rec_bankruptcies: float (nullable = true)
 |-- inq_last_6mths: float (nullable = true)
 |-- total_rec_late_fee: float (nullable = true)
 |-- mths_since_last_delinq: float (nullable = true)
 |-- mths_since_last_record: float (nullable = true)



In [46]:
loans_def_raw_df.createOrReplaceTempView("loan_defaulters")

In [47]:
loans_def_raw_df.groupBy("delinq_2yrs").agg(count("*").alias("total_count")).orderBy(desc("total_count")).show(50)

+-----------+-----------+
|delinq_2yrs|total_count|
+-----------+-----------+
|        0.0|    1838878|
|        1.0|     281335|
|        2.0|      81285|
|        3.0|      29539|
|        4.0|      13179|
|        5.0|       6599|
|        6.0|       3717|
|        7.0|       2062|
|        8.0|       1223|
|        9.0|        818|
|       10.0|        556|
|       11.0|        363|
|       12.0|        264|
|       null|        261|
|       13.0|        165|
|       14.0|        120|
|       15.0|         87|
|       16.0|         55|
|       18.0|         30|
|       17.0|         30|
|       19.0|         23|
|       20.0|         17|
|       21.0|         12|
|       22.0|          5|
|       24.0|          4|
|       26.0|          3|
|       29.0|          2|
|       23.0|          2|
|       25.0|          2|
|       3.44|          2|
|       30.0|          2|
|      16.42|          1|
|       6.52|          1|
|      21.72|          1|
|       1.41|          1|
|       39.0

In [7]:
loans_def_processed_df = loans_def_raw_df.withColumn("delinq_2yrs",col("delinq_2yrs").cast("integer")).fillna(0,subset=["delinq_2yrs"])

In [49]:
loans_def_processed_df.createOrReplaceTempView("loan_defaulters")

In [11]:
loans_def_processed_df.filter("delinq_2yrs is null").count()

0

In [12]:
loans_def_processed_df.groupBy("delinq_2yrs").agg(count("*").alias("total_count")).orderBy(desc("total_count"))

delinq_2yrs,total_count
0,1839141
1,281337
2,81285
3,29545
4,13180
5,6601
6,3719
7,2063
8,1226
9,821


In [8]:
loans_def_delinq_temp = loans_def_processed_df.withColumn("mths_since_last_delinq",col("mths_since_last_delinq").cast("integer"))

In [9]:
loans_def_delinq_temp.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- delinq_2yrs: integer (nullable = true)
 |-- delinq_amnt: float (nullable = true)
 |-- pub_rec: float (nullable = true)
 |-- pub_rec_bankruptcies: float (nullable = true)
 |-- inq_last_6mths: float (nullable = true)
 |-- total_rec_late_fee: float (nullable = true)
 |-- mths_since_last_delinq: integer (nullable = true)
 |-- mths_since_last_record: float (nullable = true)



In [51]:
loans_def_deling_temp.createOrReplaceTempView("loan_defaulters")

In [25]:
loans_def_delinq_df.count()

1106163

In [10]:
loans_def_delinq_df = loans_def_delinq_temp.select("member_id","delinq_2yrs","delinq_amnt","mths_since_last_delinq").filter("delinq_2yrs > 0 OR mths_since_last_delinq > 0")

In [26]:
loans_def_delinq_df.filter("member_id like '9cb79aa7323e81be1%'")

member_id,delinq_2yrs,delinq_amnt,mths_since_last_delinq


In [11]:
loans_def_records_enq_df = loans_def_processed_df.select("member_id").filter("pub_rec > 0.0 OR pub_rec_bankruptcies > 0.0 OR inq_last_6mths > 0.0")

In [12]:
loans_def_records_enq_df.count()

1070125

In [13]:
loans_def_delinq_df.write \
.option("header","true") \
.format("csv") \
.mode("overwrite") \
.option("path","/user/itv026931/lendingclubproject/cleaned/loans_defaulters_delinq_csv") \
.save()

In [14]:
loans_def_records_enq_df.write \
.option("header","true") \
.format("csv") \
.mode("overwrite") \
.option("path","/user/itv026931/lendingclubproject/cleaned/loans_defaulters_records_enq_csv") \
.save()

In [15]:
loans_def_delinq_df.write \
.option("header","true") \
.format("parquet") \
.mode("overwrite") \
.option("path","/user/itv026931/lendingclubproject/cleaned/loans_defaulters_delinq_parquet") \
.save()

In [16]:
loans_def_records_enq_df.write \
.option("header","true") \
.format("parquet") \
.mode("overwrite") \
.option("path","/user/itv026931/lendingclubproject/cleaned/loans_defaulters_records_enq_parquet") \
.save()